# Lab 9: Coreference

Apply a coreference solver to the first paragraph of *Alice's Adventures in Wonderland*, by Lewis Carroll, show the coreference chains and discuss them.

In [1]:
import os
import sys
from pathlib import Path

# Colab: install packages, mount Google Drive and use the Colab Notebooks/ihlt folder as root.
# Local: notebooks live in labs/, shared resources in the project root.
# This is just some setup for file organization
if "google.colab" in sys.modules:
    %pip install -q nltk fastcoref "transformers<5" "datasets<4"
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/Colab Notebooks/ihlt")
else:
    ROOT = Path.cwd().parent

NLTK_DIR = ROOT / "nltk_data"
# On Colab, keep it in the temporary disk: the models are large and Drive is slow.
HF_DIR = Path("/content/hf_cache") if "google.colab" in sys.modules else ROOT / "hf_cache"
# Download the Hugging Face models (fastcoref) into the project, not ~/.cache.
# It must be set before importing fastcoref / transformers.
os.environ["HF_HOME"] = str(HF_DIR)

# Notebook libs
import logging
import re
import warnings

# Hide the logs, warnings and progress bars of fastcoref / transformers.
warnings.filterwarnings("ignore", category=FutureWarning)

import nltk
import torch
from functools import partial

import fastcoref.modeling
from fastcoref import FCoref

logging.disable(logging.WARNING)
fastcoref.modeling.tqdm = partial(fastcoref.modeling.tqdm, disable=True)
import datasets
import transformers
datasets.disable_progress_bars()
transformers.logging.set_verbosity_error()

# NLTK packages needed by this notebook
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("gutenberg", download_dir=NLTK_DIR)

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

In [2]:
# The NLTK Gutenberg corpus contains the Project Gutenberg text of the book (carroll-alice.txt).
# The first paragraph is the one after the title of chapter I.
raw = nltk.corpus.gutenberg.raw("carroll-alice.txt")
chapter = raw[raw.index("CHAPTER I."):]
paragraph = chapter.split("\n\n")[1]
text = " ".join(paragraph.split())  # join the lines of the paragraph
print(text)

Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, 'and what is the use of a book,' thought Alice 'without pictures or conversation?'


In [3]:
# Coreference solver: fastcoref (FCoref), a fast neural model (distilled from LingMess,
# trained on OntoNotes) that works as a spaCy-compatible plugin.
# (spacy-experimental and coreferee from the slides do not support the spaCy 3.8 version used here.)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = FCoref(device=device)

prediction = model.predict(texts=[text])[0]
chains = prediction.get_clusters(as_strings=False)  # character spans (start, end)
print(f"Coreference chains: {len(chains)}\n")
for n, chain in enumerate(chains):
    print(f"Chain {n}: " + ", ".join(f"{text[s:e]!r} ({s}-{e})" for s, e in chain))

Coreference chains: 3

Chain 0: 'Alice' (0-5), 'her' (52-55), 'she' (119-122), 'her' (148-151), 'Alice' (260-265)
Chain 1: 'her sister' (52-62), 'her sister' (148-158)
Chain 2: 'the book her sister was reading' (139-170), 'it' (176-178), 'it' (215-217)


In [4]:
# The chains inside the text: each mention is marked with the number of its chain.
# Nested mentions ("her" inside "her sister") get both marks.
opens, closes = {}, {}
for n, chain in enumerate(chains):
    for s, e in chain:
        opens.setdefault(s, []).append(n)
        closes.setdefault(e, []).append(n)

marked = ""
for i, char in enumerate(text + " "):
    marked += "]" * len(closes.get(i, [])) + "".join(f"[{n}:" for n in sorted(opens.get(i, []), key=lambda n: -max(e for s, e in chains[n] if s == i)))
    marked += char
print(marked)

[0:Alice] was beginning to get very tired of sitting by [1:[0:her] sister] on the bank, and of having nothing to do: once or twice [0:she] had peeped into [2:the book [1:[0:her] sister] was reading], but [2:it] had no pictures or conversations in [2:it], 'and what is the use of a book,' thought [0:Alice] 'without pictures or conversation?' 


In [5]:
# Manual annotation of the paragraph, to check the chains.
# Each mention is given by its text and its occurrence number in the paragraph
# (as a whole word, so that "it" does not match inside "sitting").
def span(mention, occurrence=1):
    starts = [m.start() for m in re.finditer(r"\b" + re.escape(mention) + r"\b", text)]
    s = starts[occurrence - 1]
    return (s, s + len(mention))

gold = {
    "Alice": [span("Alice", 1), span("her", 1), span("she", 1), span("her", 2), span("Alice", 2)],
    "the sister": [span("her sister", 1), span("her sister", 2)],
    "the book": [span("the book her sister was reading"), span("it", 1), span("it", 2)],
}
for entity, mentions in gold.items():
    print(f"{entity:<11} " + ", ".join(f"{text[s:e]!r} ({s}-{e})" for s, e in mentions))

Alice       'Alice' (0-5), 'her' (52-55), 'she' (119-122), 'her' (148-151), 'Alice' (260-265)
the sister  'her sister' (52-62), 'her sister' (148-158)
the book    'the book her sister was reading' (139-170), 'it' (176-178), 'it' (215-217)


In [6]:
# Compare each gold chain with the predicted chain that shares the most mentions.
predicted = [set(map(tuple, chain)) for chain in chains]
for entity, mentions in gold.items():
    mentions = set(mentions)
    best = max(predicted, key=lambda chain: len(chain & mentions))
    print(f"{entity}: {len(best & mentions)} of {len(mentions)} mentions found in chain {predicted.index(best)}"
          f"   missing: {[text[s:e] for s, e in sorted(mentions - best)]}"
          f"   wrong: {[text[s:e] for s, e in sorted(best - mentions)]}")

found = set().union(*predicted)
all_gold = set().union(*map(set, gold.values()))
print(f"\nMentions in a predicted chain that are not in any gold chain: {[text[s:e] for s, e in sorted(found - all_gold)]}")

Alice: 5 of 5 mentions found in chain 0   missing: []   wrong: []
the sister: 2 of 2 mentions found in chain 1   missing: []   wrong: []
the book: 3 of 3 mentions found in chain 2   missing: []   wrong: []

Mentions in a predicted chain that are not in any gold chain: []


In [7]:
# Other noun phrases of the paragraph that are not in any chain. They are mentioned only once
# (singletons), or are not referential, so they correctly do not appear in the chains.
for np in ["the bank", "nothing", "pictures or conversations", "a book", "pictures or conversation"]:
    s, e = span(np)
    print(f"{np!r:<28} in a chain: {any((s, e) in chain for chain in predicted)}")

'the bank'                   in a chain: False
'nothing'                    in a chain: False
'pictures or conversations'  in a chain: False
'a book'                     in a chain: False
'pictures or conversation'   in a chain: False


In [8]:
# The same solver on the example sentence of the slides: two chains are expected,
# [My sister, She] and [a dog, him].
example = "My sister has a dog. She loves him."
print(model.predict(texts=[example])[0].get_clusters())

[['a dog', 'him']]


## Question

### Coreference solver

The solver is **fastcoref** (`FCoref`), a neural coreference model trained on OntoNotes that can be used as a spaCy plugin. It was chosen because spacy-experimental and coreferee, from the slides, do not support the spaCy 3.8 version used in this project. The paragraph is taken from the Project Gutenberg text of the book included in NLTK (`carroll-alice.txt`).

### Coreference chains

```
Chain 0: Alice, her, she, her, Alice
Chain 1: her sister, her sister
Chain 2: the book her sister was reading, it, it
```

In the text:

> [0:Alice] was beginning to get very tired of sitting by [1:[0:her] sister] on the bank, and of having nothing to do: once or twice [0:she] had peeped into [2:the book [1:[0:her] sister] was reading], but [2:it] had no pictures or conversations in [2:it], 'and what is the use of a book,' thought [0:Alice] 'without pictures or conversation?'

### What do you think about them?

**The chains are correct and complete for this paragraph.** Compared with a manual annotation of the paragraph, all 10 mentions of the three entities are found, in the right chain, and no chain contains a wrong mention:

- **Alice (5 of 5).** The chain links the name, the two possessive *her*, the subject pronoun *she* and the second *Alice*, which is far away from the first one (at the end of the paragraph, after the quote). The model also handles nested mentions: *her* is in the Alice chain, and *her sister*, which contains it, is in its own chain.
- **The sister (2 of 2).** The two *her sister* are correctly linked, even if this entity is never named. The same phrase could refer to two different sisters in general, so this needs the knowledge that the sister on the bank is the one reading the book.
- **The book (3 of 3).** The mention is the whole noun phrase *the book her sister was reading* (with its relative clause), and both *it* are linked to it. The second *it* (*"in it"*) is at the end of a coordination, and could also be a non-referential *it*, but here it correctly refers to the book.
- **Non-coreferent noun phrases are correctly left out.** *the bank*, *nothing* and *pictures or conversations* are mentioned only once (singletons), so they are not in any chain. Most importantly, ***a book*** in *"what is the use of a book"* is **not** linked to *the book*: it is generic (any book), not the book of the sister, even if it is the same noun. A simple string or head matching approach would have made this error.

**However, this paragraph is easy for a coreference solver**: it has only one female character with a name, the pronouns agree in gender and number with only one antecedent (*she*/*her* → Alice, *it* → the book), and it is a single sentence, so all the antecedents are close. The model is not perfect in harder cases: on the example sentence of the slides, *"My sister has a dog. She loves him."*, it finds the chain *[a dog, him]* but misses *[My sister, She]*, even though *She* has only one possible antecedent (the only female, singular noun phrase). So the good result on this paragraph should not be generalized: one paragraph (or one sentence) is too little to evaluate a coreference solver, which should be evaluated on an annotated corpus.